# Constrained Optimizer Comparison

This notebook compares AeroXAI's frozen conventional baseline controller against a constrained mixed-integer linear program (MILP).

Both methods use the same:

- simulated compressor fleet,
- receiver and air assumptions,
- one-hour synthetic demand scenario,
- nominal leakage assumption,
- initial pressure,
- pressure-safety bounds.

The optimizer is **simulation-only evidence**. It does not demonstrate measured plant savings.

The MILP includes:

- fixed-compressor ON/OFF decisions,
- VSD ON/OFF and load fraction,
- receiver pressure dynamics,
- minimum ON/OFF times,
- reserve-capacity requirement,
- startup penalty,
- overpressure penalty,
- terminal pressure requirement.

The terminal-pressure requirement prevents the optimizer from claiming artificial savings by simply draining stored compressed air at the end of the horizon.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()

if (
    not (ROOT / "ml").exists()
    and (ROOT.parent / "ml").exists()
):
    ROOT = ROOT.parent

if not (ROOT / "ml").exists():
    raise RuntimeError(
        "Could not locate repository root."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(ROOT),
    )

from ml.control.baseline import (
    BaselineControllerConfig,
    simulate_baseline,
)
from ml.optimization.scheduler import (
    OptimizationConfig,
    optimize_schedule,
)
from ml.twin.compressor import CompressorSpec
from ml.twin.physics import TwinParameters

print("Repository root:", ROOT)

## Load frozen simulation and optimization configuration

In [ ]:
with (
    ROOT
    / "configs"
    / "compressors.yaml"
).open(
    "r",
    encoding="utf-8",
) as handle:
    compressor_config = yaml.safe_load(handle)

with (
    ROOT
    / "configs"
    / "twin.yaml"
).open(
    "r",
    encoding="utf-8",
) as handle:
    twin_config = yaml.safe_load(handle)

with (
    ROOT
    / "configs"
    / "optimization.yaml"
).open(
    "r",
    encoding="utf-8",
) as handle:
    optimization_raw = yaml.safe_load(
        handle
    )["optimization"]

compressors = [
    CompressorSpec(
        id=str(item["id"]),
        kind=str(item["kind"]),
        max_mass_flow_kg_s=float(
            item["max_mass_flow_kg_s"]
        ),
        rated_power_kw=float(
            item["rated_power_kw"]
        ),
        idle_power_kw=float(
            item["idle_power_kw"]
        ),
        min_load_fraction=float(
            item["min_load_fraction"]
        ),
        min_on_seconds=float(
            item["min_on_seconds"]
        ),
        min_off_seconds=float(
            item["min_off_seconds"]
        ),
    )
    for item in compressor_config[
        "compressors"
    ]
]

pressure_config = compressor_config[
    "pressure"
]

controller_raw = compressor_config[
    "controller"
]

baseline_controller = (
    BaselineControllerConfig(
        target_bar_g=float(
            pressure_config[
                "target_bar_g"
            ]
        ),
        lower_band_bar_g=float(
            pressure_config[
                "lower_band_bar_g"
            ]
        ),
        upper_band_bar_g=float(
            pressure_config[
                "upper_band_bar_g"
            ]
        ),
        safety_min_bar_g=float(
            pressure_config[
                "safety_min_bar_g"
            ]
        ),
        safety_max_bar_g=float(
            pressure_config[
                "safety_max_bar_g"
            ]
        ),
        pressure_gain_kg_s_per_bar=float(
            controller_raw[
                "pressure_gain_kg_s_per_bar"
            ]
        ),
    )
)

air_config = twin_config["air"]
receiver_config = twin_config["receiver"]
simulation_config = twin_config[
    "simulation"
]

parameters = TwinParameters(
    volume_m3=float(
        receiver_config["volume_m3"]
    ),
    temperature_k=float(
        air_config["temperature_k"]
    ),
    gas_constant_j_per_kg_k=float(
        air_config[
            "gas_constant_j_per_kg_k"
        ]
    ),
    ambient_pressure_pa=float(
        air_config[
            "ambient_pressure_pa"
        ]
    ),
)

baseline_timestep_seconds = float(
    simulation_config[
        "timestep_seconds"
    ]
)

solver_raw = optimization_raw[
    "solver"
]

optimizer_config = OptimizationConfig(
    interval_seconds=float(
        optimization_raw[
            "interval_seconds"
        ]
    ),
    reserve_mass_flow_kg_s=float(
        optimization_raw[
            "reserve_mass_flow_kg_s"
        ]
    ),
    startup_penalty_kwh=float(
        optimization_raw[
            "startup_penalty_kwh"
        ]
    ),
    overpressure_penalty_kwh_per_bar_hour=float(
        optimization_raw[
            "overpressure_penalty_kwh_per_bar_hour"
        ]
    ),
    terminal_pressure_min_bar_g=float(
        optimization_raw[
            "terminal_pressure_min_bar_g"
        ]
    ),
    time_limit_seconds=float(
        solver_raw[
            "time_limit_seconds"
        ]
    ),
    mip_rel_gap=float(
        solver_raw[
            "mip_rel_gap"
        ]
    ),
)

nominal_leak_kg_s = float(
    compressor_config[
        "scenario"
    ][
        "nominal_leak_kg_s"
    ]
)

optimizer_config

## Recreate the frozen one-hour scenario

The baseline runs at the 1-second physics timestep. The optimizer selects commands every 60 seconds.

Because demand and compressor commands are constant inside each optimizer interval and the pressure model has a constant derivative over that interval, pressure varies linearly between interval endpoints. Therefore enforcing the safety limits at both interval endpoints also bounds the continuous trajectory within that interval.

In [ ]:
block_seconds = 15 * 60

baseline_demand = (
    [0.070] * block_seconds
    + [0.110] * block_seconds
    + [0.145] * block_seconds
    + [0.090] * block_seconds
)

optimizer_interval_seconds = int(
    optimizer_config.interval_seconds
)

if (
    block_seconds
    % optimizer_interval_seconds
    != 0
):
    raise ValueError(
        "Optimizer interval must divide "
        "the frozen 15-minute demand block."
    )

intervals_per_block = (
    block_seconds
    // optimizer_interval_seconds
)

optimizer_demand = (
    [0.070] * intervals_per_block
    + [0.110] * intervals_per_block
    + [0.145] * intervals_per_block
    + [0.090] * intervals_per_block
)

baseline_duration = (
    len(baseline_demand)
    * baseline_timestep_seconds
)

optimizer_duration = (
    len(optimizer_demand)
    * optimizer_config.interval_seconds
)

if baseline_duration != optimizer_duration:
    raise ValueError(
        "Baseline and optimizer horizons "
        "do not match."
    )

print(
    "Horizon:",
    baseline_duration,
    "seconds",
)

print(
    "Baseline steps:",
    len(baseline_demand),
)

print(
    "Optimizer intervals:",
    len(optimizer_demand),
)

## Run the frozen conventional baseline

In [ ]:
baseline = simulate_baseline(
    baseline_demand,
    leak_mass_flow_kg_s=(
        nominal_leak_kg_s
    ),
    initial_pressure_bar_g=7.0,
    parameters=parameters,
    compressors=compressors,
    controller_config=(
        baseline_controller
    ),
    timestep_seconds=(
        baseline_timestep_seconds
    ),
)

baseline_energy_kwh = float(
    baseline[
        "cumulative_energy_kwh"
    ].iloc[-1]
)

baseline_start_columns = [
    column
    for column in baseline.columns
    if column.endswith("_start")
]

baseline_start_count = int(
    baseline[
        baseline_start_columns
    ].sum().sum()
)

baseline_min_pressure = float(
    baseline[
        "pressure_bar_g"
    ].min()
)

baseline_max_pressure = float(
    baseline[
        "pressure_bar_g"
    ].max()
)

baseline_safety_violations = int(
    baseline[
        "safety_violation"
    ].sum()
)

print(
    "Baseline energy:",
    baseline_energy_kwh,
    "kWh",
)

print(
    "Baseline pressure:",
    baseline_min_pressure,
    "to",
    baseline_max_pressure,
    "bar(g)",
)

print(
    "Baseline starts:",
    baseline_start_count,
)

print(
    "Baseline safety violations:",
    baseline_safety_violations,
)

## Solve the constrained MILP

In [ ]:
optimized = optimize_schedule(
    optimizer_demand,
    leak_mass_flow_kg_s=(
        nominal_leak_kg_s
    ),
    initial_pressure_bar_g=7.0,
    parameters=parameters,
    compressors=compressors,
    target_bar_g=float(
        pressure_config[
            "target_bar_g"
        ]
    ),
    safety_min_bar_g=float(
        pressure_config[
            "safety_min_bar_g"
        ]
    ),
    safety_max_bar_g=float(
        pressure_config[
            "safety_max_bar_g"
        ]
    ),
    config=optimizer_config,
)

schedule = optimized.schedule

optimized_min_pressure = float(
    schedule[
        "pressure_bar_g"
    ].min()
)

optimized_max_pressure = float(
    schedule[
        "pressure_bar_g"
    ].max()
)

optimized_safety_violations = int(
    schedule[
        "safety_violation"
    ].sum()
)

minimum_reserve = float(
    schedule[
        "reserve_available_kg_s"
    ].min()
)

print(
    "Solver:",
    optimized.solver_message,
)

print(
    "Optimized energy:",
    optimized.energy_kwh,
    "kWh",
)

print(
    "Optimized pressure:",
    optimized_min_pressure,
    "to",
    optimized_max_pressure,
    "bar(g)",
)

print(
    "Optimized starts:",
    optimized.startup_count,
)

print(
    "Minimum available reserve:",
    minimum_reserve,
    "kg/s",
)

print(
    "Optimized safety violations:",
    optimized_safety_violations,
)

## Energy comparison

The optimization objective contains small startup and overpressure penalties, but the energy comparison below uses **electrical energy only**, so the reported saving is not an objective-function artifact.

In [ ]:
energy_saving_kwh = (
    baseline_energy_kwh
    - optimized.energy_kwh
)

energy_saving_percent = (
    energy_saving_kwh
    / baseline_energy_kwh
    * 100.0
)

print(
    "Baseline energy:",
    baseline_energy_kwh,
    "kWh",
)

print(
    "Optimized energy:",
    optimized.energy_kwh,
    "kWh",
)

print(
    "Predicted energy saving:",
    energy_saving_kwh,
    "kWh",
)

print(
    "Predicted energy saving:",
    energy_saving_percent,
    "%",
)

print(
    "MILP objective value:",
    optimized.objective_value,
)

## Pressure comparison

In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 5)
)

ax.plot(
    baseline[
        "time_seconds"
    ] / 60.0,
    baseline[
        "pressure_bar_g"
    ],
    label="Baseline",
)

ax.step(
    schedule[
        "time_seconds"
    ] / 60.0,
    schedule[
        "pressure_bar_g"
    ],
    where="post",
    label="MILP interval endpoints",
)

ax.axhline(
    float(
        pressure_config[
            "target_bar_g"
        ]
    ),
    linestyle="--",
    label="Target",
)

ax.axhline(
    float(
        pressure_config[
            "safety_min_bar_g"
        ]
    ),
    linestyle=":",
    label="Safety minimum",
)

ax.axhline(
    float(
        pressure_config[
            "safety_max_bar_g"
        ]
    ),
    linestyle=":",
    label="Safety maximum",
)

ax.set_title(
    "Baseline vs optimized receiver pressure"
)

ax.set_xlabel(
    "Time (minutes)"
)

ax.set_ylabel(
    "Pressure (bar gauge)"
)

ax.legend()

plt.show()

## Power comparison

In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 5)
)

ax.plot(
    baseline[
        "time_seconds"
    ] / 60.0,
    baseline[
        "power_kw"
    ],
    label="Baseline",
)

ax.step(
    schedule[
        "time_seconds"
    ] / 60.0,
    schedule[
        "power_kw"
    ],
    where="post",
    label="Optimized",
)

ax.set_title(
    "Baseline vs optimized compressor power"
)

ax.set_xlabel(
    "Time (minutes)"
)

ax.set_ylabel(
    "Power (kW)"
)

ax.legend()

plt.show()

## Optimized compressor schedule

In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 5)
)

time_minutes = (
    schedule[
        "time_seconds"
    ] / 60.0
)

ax.step(
    time_minutes,
    schedule[
        "fixed_1_on"
    ].astype(int),
    where="post",
    label="Fixed 1 ON",
)

ax.step(
    time_minutes,
    schedule[
        "fixed_2_on"
    ].astype(int),
    where="post",
    label="Fixed 2 ON",
)

ax.step(
    time_minutes,
    schedule[
        "vsd_1_fraction"
    ],
    where="post",
    label="VSD load fraction",
)

ax.set_title(
    "MILP compressor schedule"
)

ax.set_xlabel(
    "Time (minutes)"
)

ax.set_ylabel(
    "State / load fraction"
)

ax.legend()

plt.show()

## Write optimization evidence

This report deliberately distinguishes the simulated energy result from the real MetroPT anomaly-detection evidence.

In [ ]:
REPORT_PATH = (
    ROOT
    / "docs"
    / "optimizer_report.json"
)

report = {
    "evidence_class": "simulated",

    "method": {
        "name":
            "constrained_milp",
        "optimizer_interval_seconds":
            optimizer_config
            .interval_seconds,
        "reserve_mass_flow_kg_s":
            optimizer_config
            .reserve_mass_flow_kg_s,
        "startup_penalty_kwh":
            optimizer_config
            .startup_penalty_kwh,
        "overpressure_penalty_kwh_per_bar_hour":
            optimizer_config
            .overpressure_penalty_kwh_per_bar_hour,
        "terminal_pressure_min_bar_g":
            optimizer_config
            .terminal_pressure_min_bar_g,
        "solver_status":
            optimized.solver_status,
        "solver_message":
            optimized.solver_message,
    },

    "scenario": {
        "duration_seconds":
            baseline_duration,
        "initial_pressure_bar_g":
            7.0,
        "nominal_leak_kg_s":
            nominal_leak_kg_s,
        "demand_blocks_kg_s": [
            0.070,
            0.110,
            0.145,
            0.090,
        ],
        "block_duration_seconds":
            block_seconds,
        "safety_min_bar_g":
            float(
                pressure_config[
                    "safety_min_bar_g"
                ]
            ),
        "safety_max_bar_g":
            float(
                pressure_config[
                    "safety_max_bar_g"
                ]
            ),
    },

    "baseline": {
        "energy_kwh":
            baseline_energy_kwh,
        "minimum_pressure_bar_g":
            baseline_min_pressure,
        "maximum_pressure_bar_g":
            baseline_max_pressure,
        "startup_count":
            baseline_start_count,
        "safety_violations":
            baseline_safety_violations,
    },

    "optimized": {
        "energy_kwh":
            optimized.energy_kwh,
        "minimum_pressure_bar_g":
            optimized_min_pressure,
        "maximum_pressure_bar_g":
            optimized_max_pressure,
        "startup_count":
            optimized.startup_count,
        "overpressure_bar_hours":
            optimized
            .overpressure_bar_hours,
        "minimum_reserve_kg_s":
            minimum_reserve,
        "safety_violations":
            optimized_safety_violations,
        "objective_value":
            optimized.objective_value,
    },

    "comparison": {
        "energy_saving_kwh":
            energy_saving_kwh,
        "energy_saving_percent":
            energy_saving_percent,
    },

    "limitations": [
        (
            "All compressor specifications "
            "are simulation assumptions."
        ),
        (
            "The demand profile is synthetic."
        ),
        (
            "Leak rate is an imposed scenario "
            "input, not inferred from MetroPT."
        ),
        (
            "The compressor power model is "
            "simplified and does not model "
            "pressure-dependent efficiency."
        ),
        (
            "The optimizer uses 60-second "
            "control intervals."
        ),
        (
            "The result is predicted simulated "
            "energy performance, not measured "
            "industrial energy savings."
        ),
        (
            "AeroXAI remains advisory-only and "
            "does not override equipment control."
        ),
    ],
}

REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Optimizer report written to:",
    REPORT_PATH,
)

## Interpretation

The optimization result should be reported exactly as produced by the frozen scenario.

Do not retune the scenario to obtain a larger savings percentage.

A modest improvement is still valid evidence if it preserves:

- pressure safety,
- terminal stored-air state,
- reserve capacity,
- minimum ON/OFF behavior.

The larger simulated energy penalty from the separate leakage scenario remains a different claim from optimizer scheduling savings.